# 05. Split Strategy Comparison: Model Under an Honest Split

Refits the exact final configuration under a naive random row split and compares it against the client grouped split already fit in the clustering notebook, the same before and after check used in the sibling validation assignment.

## Setup

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# the five behavioral features the final KMeans model clusters on, shared by every
# notebook downstream of the model fit so the feature set cannot silently drift between
# the clustering, validation, and leakage audit notebooks
FEATURE_COLS = ["ctr", "avg_position", "days_since_last_update", "total_impressions_log", "engagement_rate_filled"]

model_data = pd.read_csv("../interim/model_data.csv")
train = pd.read_csv("../interim/train.csv")
test = pd.read_csv("../interim/test.csv")

SCALER = RobustScaler
k = 20
SIL_SAMPLE_SIZE = 5000

## After, the grouped split already fit in the clustering notebook

The grouped split's silhouette gets recomputed here by refitting the scaler on the saved train and test features and reusing the kmeans_cluster labels already produced in the clustering notebook, without refitting KMeans itself. The clustering does not need to be redone, only rescaled, to get a silhouette score for this split, so this stays fast and avoids any drift from the clustering notebook's result. train and test silhouette for the grouped split come out below.

In [2]:
scaler = SCALER()
X_train = scaler.fit_transform(train[FEATURE_COLS])
X_test = scaler.transform(test[FEATURE_COLS])

grouped_train_silhouette = silhouette_score(X_train, train["kmeans_cluster"].values, sample_size=SIL_SAMPLE_SIZE, random_state=42)
grouped_test_silhouette = silhouette_score(X_test, test["kmeans_cluster"].values, sample_size=SIL_SAMPLE_SIZE, random_state=42)

grouped_client_overlap = set(train["client_hash_id"]) & set(test["client_hash_id"])
print(f"GROUPED split, train silhouette: {grouped_train_silhouette:.3f}")
print(f"GROUPED split, test silhouette: {grouped_test_silhouette:.3f}")
print(f"GROUPED split clients appearing in both train and test: {len(grouped_client_overlap)}")

GROUPED split, train silhouette: 0.327
GROUPED split, test silhouette: 0.302
GROUPED split clients appearing in both train and test: 0


## Before, a naive random row split

model_data gets split by row here instead of by client, the exact same configuration gets refit, full_signal features, RobustScaler, k=20, and silhouette and client overlap get computed the same way. This is the illusion check, a naive split lets the same client's pages land on both sides, so a close train and test match under it is not evidence of generalization. The naive split lets most clients appear on both sides, and its train and test silhouette sit close together, closer than the grouped split's, for the wrong reason.

In [3]:
naive_train, naive_test = train_test_split(model_data, test_size=0.2, random_state=42)
naive_train = naive_train.copy()
naive_test = naive_test.copy()

naive_train["total_impressions_log"] = np.log1p(naive_train["total_impressions"])
naive_test["total_impressions_log"] = np.log1p(naive_test["total_impressions"])
naive_train["engagement_rate_filled"] = naive_train["engagement_rate"].fillna(0)
naive_test["engagement_rate_filled"] = naive_test["engagement_rate"].fillna(0)

scaler_naive = SCALER()
X_train_naive = scaler_naive.fit_transform(naive_train[FEATURE_COLS])
X_test_naive = scaler_naive.transform(naive_test[FEATURE_COLS])

kmeans_naive = KMeans(n_clusters=k, random_state=42, n_init=10)
naive_train_clusters = kmeans_naive.fit_predict(X_train_naive)
naive_test_clusters = kmeans_naive.predict(X_test_naive)

naive_train_silhouette = silhouette_score(X_train_naive, naive_train_clusters, sample_size=SIL_SAMPLE_SIZE, random_state=42)
naive_test_silhouette = silhouette_score(X_test_naive, naive_test_clusters, sample_size=SIL_SAMPLE_SIZE, random_state=42)

naive_client_overlap = set(naive_train["client_hash_id"]) & set(naive_test["client_hash_id"])
print(f"NAIVE split, train silhouette: {naive_train_silhouette:.3f}")
print(f"NAIVE split, test silhouette: {naive_test_silhouette:.3f}")
print(f"NAIVE split clients appearing in both train and test: {len(naive_client_overlap)} of "
      f"{model_data['client_hash_id'].nunique()} total clients")

NAIVE split, train silhouette: 0.315
NAIVE split, test silhouette: 0.310
NAIVE split clients appearing in both train and test: 47 of 50 total clients


## Before and after, side by side

Both splits get put into one table here. This is the actual before and after comparison, not two separate numbers a reader has to hold in their head. The grouped split has zero client overlap and a real gap between train and test silhouette, the naive split's closer agreement reflects leakage, not a better model.

In [4]:
comparison_before_after = pd.DataFrame({
    "split_type": ["Naive random row split", "Grouped by client split"],
    "train_silhouette": [round(naive_train_silhouette, 3), round(grouped_train_silhouette, 3)],
    "test_silhouette": [round(naive_test_silhouette, 3), round(grouped_test_silhouette, 3)],
    "train_test_gap": [
        round(naive_train_silhouette - naive_test_silhouette, 3),
        round(grouped_train_silhouette - grouped_test_silhouette, 3),
    ],
    "client_overlap": [len(naive_client_overlap), len(grouped_client_overlap)],
})
comparison_before_after

,split_type,train_silhouette,test_silhouette,train_test_gap,client_overlap
0,Naive random row split,0.315,0.310,0.005,47
1,Grouped by client split,0.327,0.302,0.025,0
